# CampusSaathi AI - RAG Data Ingestion & FAISS Index Generator
This notebook processes the ABES Engineering College FAQ PDF, generates text embeddings using Google Gemini / sentence-transformers, builds a FAISS vector index, and tests context retrieval.

In [ ]:
!pip install -q pypdf faiss-cpu google-generativeai sentence-transformers numpy

In [ ]:
import os
from pypdf import PdfReader
import faiss
import numpy as np
import google.generativeai as genai
from google.colab import files

# 1. Upload ABES FAQ PDF
print("Please upload your ABES FAQ PDF file:")
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
print(f"Processing file: {pdf_filename}")

In [ ]:
# 2. Extract Text & Create Overlapping Chunks
reader = PdfReader(pdf_filename)
chunks = []
metadata = []

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

for page_num, page in enumerate(reader.pages):
    text = page.extract_text()
    if not text:
        continue
    for i in range(0, len(text), CHUNK_SIZE - CHUNK_OVERLAP):
        chunk_text = text[i:i + CHUNK_SIZE]
        chunks.append(chunk_text)
        metadata.append({"source": pdf_filename, "page": page_num + 1})

print(f"Extracted {len(chunks)} text chunks across {len(reader.pages)} pages.")

In [ ]:
# 3. Generate Embeddings & Build FAISS Index
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(chunks, convert_to_numpy=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"FAISS index built successfully with {index.ntotal} vectors.")

In [ ]:
# 4. RAG Query with Gemini
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', 'YOUR_GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-1.5-flash')

def answer_question(question, top_k=3):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_emb, top_k)
    
    retrieved_texts = []
    sources = set()
    for idx in indices[0]:
        if idx < len(chunks):
            retrieved_texts.append(chunks[idx])
            sources.add(f"{metadata[idx]['source']} (Page {metadata[idx]['page']})")
            
    context = "\n---\n".join(retrieved_texts)
    
    prompt = f"""You are CampusSaathi AI, an official university FAQ assistant.
Answer the student's question based strictly on the provided context below.
If the context does not contain sufficient evidence to answer, state explicitly:
"I am sorry, but I do not have enough information in the official documents to answer this question."
Do NOT make up facts or hallucinate.

Context:
{context}

Question: {question}
Answer:"""
    
    response = model.generate_content(prompt)
    return response.text, list(sources)

# Test query
# ans, srcs = answer_question("What is the admission procedure for B.Tech?")
# print(ans)
# print("Sources:", srcs)